notebooks/03_gold_dimensions.py
============================================================
# NOTEBOOK 03 — Gold Layer: Dimension Tables (SCD Type 1)
# ============================================================
 Purpose : Build and maintain Star Schema dimension tables in
          the Gold layer using SCD Type 1 MERGE logic.

Tables:
  gold.dim_customer   ← silver.customers   (SCD Type 1)
   gold.dim_product    ← silver.products    (SCD Type 1)
   gold.dim_date       ← generated          (static)
  gold.dim_geography  ← silver.customers   (SCD Type 1)

 SCD Type 1 Behaviour:
  MATCHED     → overwrite changed attributes (no history kept)
  NOT MATCHED → insert new row with surrogate key
============================================================

In [0]:
import os,sys
sys.path.insert(0,os.path.abspath(os.path.join(os.getcwd(), '..')))
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
from utils.spark_utils import(
    add_silver_audit_columns,
    write_delta_overwrite,
    write_delta_append,
    dedup,
    drop_nulls,
    
)
from config.config import (
    SILVER_CUSTOMERS, SILVER_PRODUCTS,
    DIM_CUSTOMER, DIM_PRODUCT, DIM_DATE, DIM_GEOGRAPHY,
    GOLD_PATH, DIM_DATE_START, DIM_DATE_END
)

from utils.dq_checks import(
    standard_checks,
)
from utils.scd_utils import(
    add_surrogate_key,
)

In [0]:
 logger.info("Building dim_customer (SCD Type 1)...")

In [0]:
silver_df = spark.read.format("delta")\
    .load(SILVER_CUSTOMERS)

In [0]:
staging = silver_df.select(
    col("customer_id").alias("customer_bk"),
    "first_name", "last_name", "full_name",
    "email", "phone",
    "city", "state", "country", "zip_code",
    "customer_segment",
    "registration_date"
)

In [0]:
staging = add_surrogate_key(staging, "customer_bk", "customer_sk")

In [0]:
if spark.catalog.tableExists("gold.dim_customer"):

    scd_type1_merge(
        staging,
        target_table="gold.dim_customer",
        key_cols=["customer_bk"],
        cdc="updated_at"
    )
else:
    write_delta_overwrite(staging,'/Volumes/salesdw/gold/gold_data/dim_customers')

logger.info("Building dim_product (SCD Type 1)...")

In [0]:
 optimize_table(spark, DIM_CUSTOMER, z_order_cols=["customer_bk"])

In [0]:
logger.info("Building products_products (SCD Type 1)...")

In [0]:
silver_df = spark.read.format("delta")\
    .load(SILVER_PRODUCTS)
silver.select(
            F.col("product_id").alias("product_bk"),
            "product_name", "category", "sub_category",
            "brand", "sku", "supplier_id",
            "unit_cost", "unit_price", "gross_margin_pct",
            "weight_kg", "is_active", "created_date"
        )


In [0]:
staging = add_surrogate_key(staging, "products_bk", "products_sk")

In [0]:
if spark.catalog.tableExists("gold.dim_products"):

    scd_type1_merge(
        staging,
        target_table="gold.dim_products",
        key_cols=["customer_bk"],
        cdc="updated_at"
    )
else:
    write_delta_overwrite(staging,'/Volumes/salesdw/gold/gold_data/dim_products')

logger.info("Building dim_product (SCD Type 1)...")

In [0]:
 optimize_table(spark, DIM_PRODUCTS, z_order_cols=["customer_bk"])

**dim_date**

In [0]:
logger.info("Building dim_date (SCD Type 1)...")

In [0]:
  # Generate date range as a DataFrame
date_df = spark.sql(f"""
    SELECT sequence(
            to_date('{DIM_DATE_START}'),
            to_date('{DIM_DATE_END}'),
            interval 1 day
    ) AS date_array
    """).withColumn("full_date", explode(col("date_array"))).select("full_date")

In [0]:
    dim_date = date_df
        .withColumn("date_key",         date_format(col("full_date"), "yyyyMMdd").cast(IntegerType()))
        .withColumn("year",              year(col("full_date")))
        .withColumn("quarter",           quarter(F.col("full_date")))
        .withColumn("quarter_name",      concat(lit("Q"), quarter(F.col("full_date"))))
        .withColumn("month",             month(col("full_date")))
        .withColumn("month_name",        date_format(col("full_date"), "MMMM"))
        .withColumn("month_abbrev",      date_format(col("full_date"), "MMM"))
        .withColumn("week_of_year",      weekofyear(col("full_date")))
        .withColumn("day_of_month",      dayofmonth(col("full_date")))
        .withColumn("day_of_week",       dayofweek(col("full_date")))
        .withColumn("day_name",          date_format(col("full_date"), "EEEE"))
        .withColumn("day_abbrev",        date_format(col("full_date"), "EEE"))
        .withColumn("is_weekend",        (dayofweek(col("full_date")).isin(1, 7)).cast(BooleanType()))
        .withColumn("is_weekday",        (dayofweek(col("full_date")).isin(1, 7)).cast(BooleanType()))
        .withColumn("year_month",        date_format(col("full_date"), "yyyy-MM"))
        .withColumn("year_quarter",
                    concat(year(col("full_date")), lit("-Q"), quarter(col("full_date"))))
        .withColumn("fiscal_year",
                    when(month(col("full_date")) >= 7,
                        year(col("full_date")) + 1)
                     .otherwise(year(col("full_date"))))
        .withColumn("fiscal_quarter",
                    when(month(col("full_date")).isin(7, 8, 9), lit("FQ1"))
                     .when(month(col("full_date")).isin(10, 11, 12), lit("FQ2"))
                     .when(month(col("full_date")).isin(1, 2, 3), lit("FQ3"))
                     .otherwise(lit("FQ4")))
    

    final_cols = [
        "date_key", "full_date", "year", "quarter", "quarter_name",
        "month", "month_name", "month_abbrev", "week_of_year",
        "day_of_month", "day_of_week", "day_name", "day_abbrev",
        "is_weekend", "is_weekday", "year_month", "year_quarter",
        "fiscal_year", "fiscal_quarter"
    ]
    dim_date= dim_date.select(*final_cols),

In [0]:
 write_delta_overwrite(
        dim_date
        '/Volumes/salesdw/gold/gold_data/dim_products'
        
    )
    optimize_table(spark, DIM_DATE, z_order_cols=["date_key"])
    logger.info(f" dim_date complete → {DIM_DATE}")

**Fact Table**

In [0]:
oredrs=read_delta("/Volumes/salesdw/silver/silver_data/silver_orders")
  # ---------------------------------------------------------------
    # 2. Load dimension tables (select only required columns)
    # ---------------------------------------------------------------
  dim_customer = read_delta("/Volumes/salesdw/gold/gold_data/dim_customers").select(
    "customer_bk",
    "customer_sk"
)

dim_product = read_delta("/Volumes/salesdw/gold/gold_data/dim_products").select(
    "product_bk",
    "product_sk"
)

dim_date = read_delta("/Volumes/salesdw/gold/gold_data/dim_date").select(
    "full_date",
    "date_key"
)

silver_products=read_delta("/Volumes/salesdw/silver/silver_data/silver_products").select("unit_price").alias("product_unit_price")


In [0]:
orders=oredrs.join(dim_customer, oredrs.customer_id==dim_customer.customer_bk, "left").join(dim_product, oredrs.product_id==dim_product.product_bk, "left").join(dim_date, oredrs.order_date==dim_date.full_date, "left").join(silver_products, orders.product_id==silver_products.product_id, "left")


In [0]:
 df_fact = 
        orders
        .withColumn("cogs",
                    round(col("quantity") * F.col("product_unit_cost"), 2))
        .withColumn("gross_profit",
                    round(F.col("net_revenue") - col("cogs"), 2))
        .withColumn("gross_profit_pct",
                    when(col("net_revenue") != 0,
                           round(F.col("gross_profit") / col("net_revenue") * 100, 2))
                     .otherwise(lit(0.0)))
    

In [0]:
  final_cols = [
        # Surrogate Keys (FKs to dimensions)
        col("customer_sk"),
        col("product_sk"),
        col("order_date_sk"),
        col("ship_date_sk"),
        col("geography_sk"),

        # Degenerate Dimensions (stored directly on fact)
        col("order_id"),
        col("order_line_id"),
        col("payment_method"),
        col("order_channel"),
        col("order_status"),

        # Measures
        col("quantity"),
        col("unit_price"),
        col("product_unit_cost").alias("unit_cost"),
        col("discount_pct"),
        col("discount_amount"),
        col("gross_revenue"),
        col("net_revenue"),
        col("cogs"),
        col("gross_profit"),
        col("gross_profit_pct"),
        col("days_to_ship"),

        # Audit
        col("order_date"),         # actual date (for partitioning)
        col("source_system"),
        col("batch_id"),
        current_timestamp().alias("created_at"),
        current_timestamp().alias("updated_at"),
    ]

    fact_final = fact.select(*final_cols)

In [0]:
if table_exists(spark, FACT_SALES):

    scd_type1_merge(
        df=fact_final,
        target_table=FACT_SALES,
        key_cols=["order_line_id"],
        cdc_col="last_updated"      # replace with your CDC column
    )

else:

    write_delta_overwrite(
        df=fact_final,
        table_name=FACT_SALES,
        path="/Volumes/salesdw/gold/gold_data/fact_sales",
        partition_by=["order_date"]
    )

optimize_table(
    FACT_SALES,
    z_order_cols=["customer_sk", "product_sk", "order_date_sk"]
)

logger.info(f"fact_sales complete → {FACT_SALES}")

In [0]:
    run_standard_checks(
        fact_final,"FACT_SALES",
        not_null_cols=["order_line_id", "customer_sk", "product_sk", "order_date_sk"],
        dedup_key_cols=["order_line_id"]
    )
